### Imports

In [0]:
import pyspark.sql.functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### Declarations/Variables/Parameters

In [0]:
# ---------- Analysis window ----------
START_DATE     = "2022-01-01"          # inclusive
END_DATE       = "2026-11-09"          # exclusive  → YTD through 2026-11-08
YTD_END_MMDD   = "11-08"               # same-period cutoff applied to prior years for YoY
YEARS          = [2022, 2023, 2024, 2025, 2026]
BASELINE_PAIRS = [(2022, 2023), (2023, 2024), (2024, 2025), (2025, 2026)]  # (baseline, projected)
COVID_END      = "2022-03-31"          # option (b): keep 2022, flag Q1 as COVID-affected

# ---------- Tables (rulebook §2) ----------
CAT     = "cpx_dataanalytics_gold.customer360"
T_TRANS = f"{CAT}.c360_f_transaction"
T_LOC   = f"{CAT}.cpx_d_location"
T_ITEM  = f"{CAT}.cpx_d_item"

# ---------- LBE scope (BR-012) ----------
LBE_LOC_TYPE_LIKE = "Restaurant%"
LBE_BRANDS        = ["Rec Room", "Playdium"]
BRAND_SHORT       = {"Rec Room": "TRR", "Playdium": "PDM"}

# ---------- Definitions ----------
DATE_COL = "Transaction_DateTime"     
TXN_KEY  = ["LocationId", "txn_date", "Transaction_Number"]   

 - Scene+ transaction = any line in the transaction with an identified CDE_ID (BR-013)
- Revenue          = SUM(Net_Revenue), refunds netted (negative lines included)
- Yield            = revenue / transactions
- Week             = Mon–Sun (Spark date_trunc('week') is Monday-based)
- Same-store       = venue has transactions across the full baseline AND comparison period

In [0]:
# ---------- Revenue scope (BR-042) — INCLUDE lists; everything else dropped ----------
GAMES_CLASSES = [
    "TRR - AMUSEMENT - GAMES",
    "TRR - AMUSEMENT - PHYSICAL GAMES",
    "TRR - PROMO - AMUSEMENT",
]
FB_CLASSES = [
    # Food
    "TRR - MENU FOOD - APP", "TRR - MENU FOOD - SANDWICHES", "TRR - MENU FOOD - ENTRÉE",
    "TRR - MENU FOOD - PIZZA", "TRR - MENU FOOD - KIDS", "TRR - MENU FOOD - POUTINE",
    "TRR - MENU FOOD - DESSERT", "TRR - MENU FOOD - SALAD", "TRR - MENU FOOD - BRUNCH",
    "TRR - MENU FOOD - DONUTS", "PDM - MENU FOOD - CANDY", "TRR - PROMO - FOOD",
    "TRR - MOD MENU - ADD-ON", "TRR - MOD MENU - NO VALUE",
    # Beverage
    "TRR - MENU ALC BEV - DRAUGHT BEER", "TRR - MENU ALC BEV - LIQUOR",
    "TRR - MENU ALC BEV - COCKTAIL", "TRR - MENU ALC BEV - WINE",
    "TRR - MENU BEV - NA COLD BEV", "TRR - MENU BEV - NA HOT BEV",
    "TRR - INV ALC BEV - CAN", "TRR - INV ALC BEV - BOTTLE", "TRR - INV ALC BEV - WINE",
    "TRR - INV ALC BEV - LIQUOR", "TRR - INV BEV - NON-ALC",
    # LBE inventory items
    "LBE - INV  Dry Food", "LBE - INV Dairy", "LBE - INV Meat",
    "LBE - INV Produce", "LBE - INV Bread",
]
CATEGORY_MAP = {**{c: "Games" for c in GAMES_CLASSES}, **{c: "F&B" for c in FB_CLASSES}}

found = {r[0] for r in spark.table(T_ITEM).select("Item_Class_Description1").distinct().collect()}
missing = [c for c in CATEGORY_MAP if c not in found]
print("Missing classes:", missing if missing else "None ✅")

In [0]:
df_base = spark.table("data_exploration_hub.marketing_hub.lbe_review_base_hourly").toPandas()
df_base = spark.table("data_exploration_hub.marketing_hub.lbe_review_base_hourly").toPandas()
df_base[["revenue", "fb_revenue", "games_revenue"]] = df_base[["revenue", "fb_revenue", "games_revenue"]].astype(float)   # Decimal → float


df_base["txn_date"]      = pd.to_datetime(df_base["txn_date"])
df_base["year"]          = df_base["txn_date"].dt.year
df_base["month"]         = df_base["txn_date"].dt.to_period("M").dt.to_timestamp()
df_base["week_start"]    = df_base["txn_date"] - pd.to_timedelta(df_base["txn_date"].dt.weekday, unit="D")  # Mon
df_base["brand"]         = df_base["LC_Trade_Brand_Description"].map(BRAND_SHORT)
df_base["in_ytd_window"] = df_base["txn_date"].dt.strftime("%m-%d") <= YTD_END_MMDD
df_base["covid_flag"]    = df_base["txn_date"] <= COVID_END
print(df_base.shape)

In [0]:
# ============================================================
# Cell 4 — Venue trading windows + same-store flags
# ============================================================
traded = df_base.loc[df_base["n_txn"] > 0, ["LocationId", "brand", "Location_Name", "txn_date"]]

# First / last trading day per venue (from data, not Opening_Date)
venue = (traded.groupby(["LocationId", "brand", "Location_Name"])["txn_date"]
         .agg(first_txn="min", last_txn="max").reset_index())

# Longest gap between trading days (catches temporary closures)
d = traded[["LocationId", "txn_date"]].drop_duplicates().sort_values(["LocationId", "txn_date"])
d["gap_days"] = d.groupby("LocationId")["txn_date"].diff().dt.days
venue = venue.merge(d.groupby("LocationId")["gap_days"].max().rename("max_gap_days").reset_index(),
                    on="LocationId")

# Same-store per (baseline, projected) pair: traded from start of baseline to end of projected period
GRACE = pd.Timedelta(days=7)
def period_end(y):
    return pd.Timestamp(f"{y}-{YTD_END_MMDD}") if y == 2026 else pd.Timestamp(f"{y}-12-31")

for b, p in BASELINE_PAIRS:
    venue[f"ss_{b}_{p}"] = ((venue["first_txn"] <= pd.Timestamp(f"{b}-01-01") + GRACE) &
                            (venue["last_txn"]  >= period_end(p) - GRACE))

display(venue.sort_values(["brand", "first_txn"]))

In [0]:
# ============================================================
# Cell 5 — Transaction volume & yield views
# Period: Week / Month / Year / Year_SamePeriod (Jan 1–Nov 8)
# Level:  Location / Brand / LBE      · Total + Scene+
# ============================================================
df_base["lbe"]         = "LBE"
df_base["n_txn_scene"] = df_base["n_txn"]   * df_base["is_scene"]
df_base["rev_scene"]   = df_base["revenue"] * df_base["is_scene"]

PERIODS = {"Week": "week_start", "Month": "month", "Year": "year"}
LEVELS  = {"Location": ["brand", "LocationId", "Location_Name"], "Brand": ["brand"], "LBE": ["lbe"]}

def safe_div(a, b):
    return a / b.where(b > 0)

def build_view(df, period_col, level_cols):
    v = (df.groupby([period_col] + level_cols, as_index=False)
           .agg(n_txn=("n_txn", "sum"), revenue=("revenue", "sum"),
                n_txn_scene=("n_txn_scene", "sum"), rev_scene=("rev_scene", "sum")))
    v["yield"]           = safe_div(v["revenue"],   v["n_txn"])
    v["yield_scene"]     = safe_div(v["rev_scene"], v["n_txn_scene"])
    v["scene_txn_share"] = safe_div(v["n_txn_scene"], v["n_txn"])
    v["scene_rev_share"] = safe_div(v["rev_scene"],   v["revenue"])
    return v

views = {}
for p, pc in PERIODS.items():
    for l, lc in LEVELS.items():
        views[(p, l)] = build_view(df_base, pc, lc)
# Same-period yearly (Jan 1 – Nov 8 every year) for like-for-like YoY
for l, lc in LEVELS.items():
    views[("Year_SamePeriod", l)] = build_view(df_base[df_base["in_ytd_window"]], "year", lc)

# Partial-period flags
for l in LEVELS:
    views[("Week", l)]["partial"]  = views[("Week", l)]["week_start"] < pd.Timestamp("2022-01-03")   # Jan 1–2, 2022
    views[("Month", l)]["partial"] = views[("Month", l)]["month"] == pd.Timestamp("2026-11-01")      # 8 days
    views[("Year", l)]["partial"]  = views[("Year", l)]["year"] == 2026                              # YTD

display(views[("Year_SamePeriod", "Brand")])

# Transaction Volume Analysis

In [0]:
# ============================================================
# Cell 7 — Story 1: LBE overall · Transactions vs Yield (deck-ready)
# ============================================================
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

sns.set_theme(style="whitegrid", context="talk", font_scale=0.75,
              rc={"axes.spines.top": False, "axes.spines.right": False,
                  "axes.edgecolor": "#c3c2b7", "grid.color": "#ebeae4"})
BLUE, ORANGE, GRAY, INK, MUTED, SHADE = "#2a78d6", "#eb6834", "#c9c7bf", "#1f1f1d", "#6b6a66", "#f3f2ee"

# ---- Data: monthly, drop partial Nov 2026 ----
m = (views[("Month", "LBE")].loc[lambda x: ~x["partial"]]
     .sort_values("month").set_index("month"))
roll_txn, roll_rev = m["n_txn"].rolling(12, center=True), m["revenue"].rolling(12, center=True)
m["txn_trend"]   = roll_txn.mean()
m["yield_trend"] = roll_rev.sum() / roll_txn.sum()          # ratio of sums

BASE_DATE = pd.Timestamp("2023-01-01")      # comparison base for trend % (2022 = COVID-distorted)

def trend_panel(ax, actual, trend, color, ylabel, tick_fmt, label_fmt, title):
    ax.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2022-03-31"), color=SHADE, zorder=0)
    ax.plot(actual.index, actual, color=GRAY, lw=1.4, zorder=1)
    ax.plot(trend.index, trend, color=color, lw=3.2, solid_capstyle="round", zorder=2)
    t = trend.dropna()
    t = t[t.index >= BASE_DATE]
    x0, y0, x1, y1 = t.index[0], t.iloc[0], t.index[-1], t.iloc[-1]
    ax.scatter([x0, x1], [y0, y1], s=70, color=color, edgecolor="white", linewidth=2, zorder=3)
    ax.annotate(label_fmt(y0), (x0, y0), xytext=(-10, 12), textcoords="offset points",
                ha="right", fontsize=12, color=color, fontweight="bold")
    ax.annotate(f"{label_fmt(y1)}  ({(y1 / y0 - 1) * 100:+.0f}%)", (x1, y1), xytext=(10, 12),
                textcoords="offset points", ha="left", fontsize=12, color=color, fontweight="bold")
    ax.set_title(title, loc="left", fontsize=13, color=INK, fontweight="bold", pad=8)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=11)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: tick_fmt(v)))
    ax.legend(handles=[Line2D([], [], color=GRAY, lw=1.4, label="Monthly actual"),
                       Line2D([], [], color=color, lw=3.2, label="Trend (12-month avg)")],
              loc="upper left", frameon=False, ncol=2, fontsize=10)
    return (y1 / y0 - 1) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8.5), sharex=True)
txn_chg = trend_panel(ax1, m["n_txn"], m["txn_trend"], BLUE, "Transactions",
                      lambda v: f"{v/1e3:.0f}K", lambda v: f"{v/1e3:.0f}K", "Transactions per month")
yld_chg = trend_panel(ax2, m["yield"], m["yield_trend"], ORANGE, "$ per transaction",
                      lambda v: f"${v:.0f}", lambda v: f"${v:.2f}", "Yield (average spend per transaction)")

ax1.text(pd.Timestamp("2022-02-15"), ax1.get_ylim()[0] + 0.04 * np.ptp(ax1.get_ylim()),
         "COVID\nrestrictions", ha="center", fontsize=9, color=MUTED)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax2.set_xlim(m.index.min() - pd.Timedelta(days=20), m.index.max() + pd.Timedelta(days=170))
ax2.set_xlabel("")

fig.text(0.01, 0.995, f"Yield up {yld_chg:+.0f}% while transactions {txn_chg:+.0f}% — "
         "growth is coming from bigger checks, not more visits",
         fontsize=16, fontweight="bold", color=INK, va="top")
fig.text(0.01, 0.955, "All LBE venues (Rec Room + Playdium) · Jan 2022 – Oct 2026 · "
         "trend = centred 12-month rolling average", fontsize=11, color=MUTED, va="top")
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

# ---- Supporting table: same period (Jan 1–Nov 8) each year ----
y = views[("Year_SamePeriod", "LBE")].sort_values("year")
tbl1 = pd.DataFrame({
    "Year (Jan 1–Nov 8)":  y["year"].astype(str),
    "Transactions":        y["n_txn"].astype(int),
    "Transactions YoY %":  (y["n_txn"].pct_change() * 100).round(1),
    "Yield ($/txn)":       y["yield"].round(2),
    "Yield YoY %":         (y["yield"].pct_change() * 100).round(1),
    "Revenue YoY %":       (y["revenue"].pct_change() * 100).round(1),
})
display(tbl1)

- Guests are spending more per visit, but visiting less. Average spend per transaction rose every year, from $55.86 to $66.19 (+18%).
- Transaction volume has been flat for four years. It swings ±10–14% year to year, and 2026 YTD (1.51M) is back to 2022 levels.
- Higher spend isn’t enough to offset fewer visits. In 2026 YTD, transactions are down 10.7%, so revenue is down 6.7% despite a 4.6% higher check.

**Bottom line:** growth has come from bigger checks, not more visits. Volume is the constraint.

In [0]:
# ============================================================
# Cell 8 — Story 2: By brand · Transactions vs Yield
# ============================================================
BRAND_NAME = {"TRR": "The Rec Room", "PDM": "Playdium"}
mb = views[("Month", "Brand")].loc[lambda x: ~x["partial"]].sort_values("month")

fig, axes = plt.subplots(2, 2, figsize=(16, 8.5), sharex=True)
chg = {}
for col, br in enumerate(["TRR", "PDM"]):
    s = mb[mb["brand"] == br].set_index("month")
    rt, rr = s["n_txn"].rolling(12, center=True), s["revenue"].rolling(12, center=True)
    s = s.assign(txn_trend=rt.mean(), yield_trend=rr.sum() / rt.sum())

    t = trend_panel(axes[0, col], s["n_txn"], s["txn_trend"], BLUE, "Transactions",
                    lambda v: f"{v/1e3:.0f}K", lambda v: f"{v/1e3:.0f}K",
                    f"{BRAND_NAME[br]} — transactions per month")
    y = trend_panel(axes[1, col], s["yield"], s["yield_trend"], ORANGE, "$ per transaction",
                    lambda v: f"${v:.0f}", lambda v: f"${v:.2f}",
                    f"{BRAND_NAME[br]} — yield")
    chg[br] = (t, y)

for ax in axes.flat:                                   # one legend per row is enough
    if ax not in (axes[0, 0], axes[1, 0]):
        ax.get_legend().remove()
for ax in axes[1]:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    ax.set_xlim(mb["month"].min() - pd.Timedelta(days=20), mb["month"].max() + pd.Timedelta(days=200))
    ax.set_xlabel("")

fig.text(0.01, 0.995,
         f"Same pattern in both brands — Rec Room: yield {chg['TRR'][1]:+.0f}%, transactions {chg['TRR'][0]:+.0f}%  ·  "
         f"Playdium: yield {chg['PDM'][1]:+.0f}%, transactions {chg['PDM'][0]:+.0f}%",
         fontsize=15, fontweight="bold", color=INK, va="top")
fig.text(0.01, 0.958, "All venues incl. new openings · Jan 2022 – Oct 2026 · "
         "trend = centred 12-month rolling average · shaded = COVID restrictions",
         fontsize=11, color=MUTED, va="top")
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

# ---- Supporting table: same period (Jan 1–Nov 8) each year, by brand ----
yb = views[("Year_SamePeriod", "Brand")].sort_values(["brand", "year"])
g = yb.groupby("brand")
tbl2 = pd.DataFrame({
    "Brand":               yb["brand"].map(BRAND_NAME),
    "Year (Jan 1–Nov 8)":  yb["year"].astype(str),
    "Transactions":        yb["n_txn"].astype(int),
    "Transactions YoY %":  (g["n_txn"].pct_change() * 100).round(1),
    "Yield ($/txn)":       yb["yield"].round(2),
    "Yield YoY %":         (g["yield"].pct_change() * 100).round(1),
    "Revenue YoY %":       (g["revenue"].pct_change() * 100).round(1),
})
display(tbl2)

- **The Rec Room (about 80% of volume)**: average spend is up 19% ($58 → $69) while transactions are flat (−2%). All of the growth came from guests spending more per visit, and the spend trend is still climbing.
- **Playdium**: average spend is up 10% ($47 → $52), but it has plateaued since late 2024. Transactions are up 9%. This likely comes from the new venues (Fairview Dec 2024, Vaughan 2026), which Story 3 will test.
- **Implication**: Rec Room is leaning on price, and Playdium’s spend lever has stalled. Neither brand shows organic growth in visits so far.

In [0]:
# ============================================================
# Cell 9 — Scene+ proxy test results: transactions per visit (Jan 1–Nov 8, base 2023)
# ============================================================
df_proxy = pd.DataFrame({
    "brand":            ["Playdium"] * 4 + ["Rec Room"] * 4,
    "yr":               [2023, 2024, 2025, 2026] * 2,
    "txns_per_visit":   [1.286, 1.283, 1.282, 1.290, 1.236, 1.231, 1.226, 1.240],
    "total_txns_idx":   [100.0, 83.6, 90.5, 91.1, 100.0, 89.5, 99.1, 86.4],
    "scene_txns_idx":   [100.0, 107.1, 129.8, 130.9, 100.0, 110.8, 128.5, 120.6],
    "scene_visits_idx": [100.0, 107.3, 130.2, 130.5, 100.0, 111.3, 129.5, 120.2],
})
display(df_proxy)

In [0]:
# ============================================================
# Cell 10 — Venue view: 2×2 by year (vs best year 2023)
# x: transactions or visits · revenue: total, gaming or F&B
# Depends on: df_base, venue (Cell 4), INK/MUTED (Cell 7), df_proxy (Cell 9)
# ============================================================
import textwrap

BRAND_NAME  = {"TRR": "The Rec Room", "PDM": "Playdium"}
PROXY_BRAND = {v: k for k, v in BRAND_SHORT.items()}           # TRR → "Rec Room", PDM → "Playdium"
HEAT_CMAP   = sns.diverging_palette(15, 150, s=75, l=55, as_cmap=True)   # total:  red → green
GAME_CMAP   = sns.diverging_palette(30, 265, s=80, l=55, as_cmap=True)   # gaming: orange → purple
FB_CMAP     = sns.diverging_palette(350, 220, s=80, l=55, as_cmap=True)  # F&B:    pink → blue

# Games coded as 'Unknown' upstream in these windows → unclassified revenue counted as gaming
BREAK_WINDOWS = {"TRR": ("2023-07-01", "2024-06-30"), "PDM": ("2024-02-01", "2024-06-30")}

venue["cohort"] = np.where(venue["first_txn"] > pd.Timestamp("2022-01-07"), "New", "Existing")
short = lambda s: s.str.replace(r"^(The Rec Room |TRR |Playdium )", "", regex=True)

def cohort_data(br):
    """Same-period (Jan 1–Nov 8) rows for one brand, with venue cohort and gap-adjusted gaming revenue."""
    d = (df_base[(df_base["brand"] == br) & df_base["in_ytd_window"]]
         .merge(venue[["LocationId", "cohort"]], on="LocationId"))
    s, e = BREAK_WINDOWS[br]
    unclassified = d["revenue"] - d["fb_revenue"] - d["games_revenue"]
    d["games_adj"] = d["games_revenue"] + np.where(d["txn_date"].between(s, e), unclassified, 0)
    return d

def venue_order(br, y0=2023, y1=2026):
    """Table row order: existing venues by cumulative txn change (worst first), new venues last."""
    d = cohort_data(br)
    pv = d.groupby(["Location_Name", "cohort", "year"])["n_txn"].sum().unstack("year")
    key = (pv[y1] / pv[y0] - 1).rename("chg").reset_index()
    key["label"]  = short(key["Location_Name"]) + np.where(key["cohort"] == "New", " (new)", "")
    key["is_new"] = key["cohort"] == "New"
    return key.sort_values(["is_new", "chg", "label"])["label"].tolist()

def panel_title(ax, text):
    ax.set_title(textwrap.fill(text, 70), loc="left", fontsize=13, fontweight="bold", color=INK, pad=10)

X_MODES = {
    "txn":    dict(col="n_txn",  label="Transactions (Jan 1–Nov 8)",     name="Transactions",
                   left="fewer transactions", note="yield = revenue ÷ transactions"),
    "visits": dict(col="visits", label="Estimated visits (Jan 1–Nov 8)", name="Est. visits",
                   left="fewer visits",
                   note="visits = transactions ÷ Scene+ transactions per visit (brand × year)"),
}
REV_MODES = {
    "total": dict(col="revenue", cmap=HEAT_CMAP, name="Revenue", short="revenue",
                  y={"txn": "Yield ($ per transaction)", "visits": "Spend per visit ($)"},
                  up={"txn": "higher spend per transaction", "visits": "higher spend per visit"}),
    "games": dict(col="games_adj", cmap=GAME_CMAP, name="Gaming revenue", short="gaming",
                  y={"txn": "Gaming $ per transaction", "visits": "Gaming spend per visit ($)"},
                  note2="Gaming includes unclassified lines during the 2023–24 coding gap · excludes kiosk sales"),
    "fb":    dict(col="fb_revenue", cmap=FB_CMAP, name="F&B revenue", short="F&B", step=2e6,
                  y={"txn": "F&B $ per transaction", "visits": "F&B spend per visit ($)"},
                  note2="F&B excludes G&E and COMP items · not affected by the 2023–24 coding gap"),
}

def venue_abs_by_year(br, x="visits", rev="total", y0=2023, y1=2026):
    xm, rm = X_MODES[x], REV_MODES[rev]
    xc, ylab = xm["col"], rm["y"][x]
    d = cohort_data(br)
    g = (d[d["year"] >= y0].groupby(["Location_Name", "cohort", "year"])
           [["n_txn", "revenue", "games_adj", "fb_revenue"]].sum().reset_index())

    tpv = df_proxy[df_proxy["brand"] == PROXY_BRAND[br]].set_index("yr")["txns_per_visit"]
    g["visits"] = g["n_txn"] / g["year"].map(tpv)                # estimated visits
    g["rev"]    = g[rm["col"]]
    g["y"]      = g["rev"] / g[xc]                               # $ per transaction or per visit
    g["venue"]  = short(g["Location_Name"]) + np.where(g["cohort"] == "New", " (new)", "")
    first_rev = g.sort_values("year").groupby("Location_Name")["rev"].transform("first")
    g["rev_vs_base"] = (g["rev"] / first_rev - 1) * 100           # vs y0, or vs first year for new venues

    # Shared axes, bubble scale and iso-revenue levels
    xr = (0, g[xc].max() * 1.15)
    yr = (g["y"].min() * 0.9, g["y"].max() * 1.08)
    smax = g["rev"].max()
    step = rm.get("step") or (5e6 if smax > 10e6 else (2e6 if smax > 4e6 else 1e6))
    levels = np.arange(step, smax * 1.3, step)
    xs = np.linspace(xr[1] * 0.02, xr[1], 400)

    years = list(range(y0, y1 + 1))
    fig, axes = plt.subplots(2, 2, figsize=(18, 15), sharex=True, sharey=True)
    for ax, yr_i in zip(axes.flat, years):
        h = g[g["year"] == yr_i]
        for lv in levels:                                          # iso-revenue: x × y = revenue
            ys = lv / xs
            ok = (ys >= yr[0]) & (ys <= yr[1])
            ax.plot(xs[ok], ys[ok], color="#c3c2b7", lw=1, ls=":", zorder=0)
            if ok.any():
                ax.text(xs[ok][-1], ys[ok][-1], f"${lv/1e6:.0f}M ", fontsize=9, color=MUTED,
                        ha="right", va="bottom")
        for mk, sub in [("o", h["cohort"] == "Existing"), ("D", h["cohort"] == "New")]:
            if sub.any():
                sc = ax.scatter(h.loc[sub, xc], h.loc[sub, "y"],
                                s=1800 * h.loc[sub, "rev"] / smax + 80, marker=mk,
                                c=h.loc[sub, "rev_vs_base"], cmap=rm["cmap"], vmin=-30, vmax=30,
                                edgecolor="white", linewidth=1.2, alpha=0.9, zorder=3)
        for _, row in h.iterrows():
            ax.annotate(row["venue"], (row[xc], row["y"]), xytext=(9, -3),
                        textcoords="offset points", fontsize=9, color=INK, va="center")
        ex = h[h["cohort"] == "Existing"]
        k = int((ex["rev_vs_base"] < 0).sum())
        sub_txt = "baseline (best year)" if yr_i == y0 else f"{k} of {len(ex)} existing venues below {y0}"
        amt = f"${h['rev'].sum()/1e6:,.0f}M" if rev == "total" else f"${h['rev'].sum()/1e6:,.1f}M"
        panel_title(ax, f"{yr_i}{' YTD' if yr_i == y1 else ''}: {rm['short']} {amt} · {sub_txt}")
        ax.set_xlim(*xr); ax.set_ylim(*yr)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1e3:,.0f}K"))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:.0f}"))

    for ax in axes[1]:
        ax.set_xlabel(xm["label"], color=MUTED, fontsize=11)
    for ax in axes[:, 0]:
        ax.set_ylabel(ylab, color=MUTED, fontsize=11)

    plt.subplots_adjust(top=0.92, right=0.90, hspace=0.18, wspace=0.06)
    cax = fig.add_axes([0.92, 0.30, 0.012, 0.40])                 # dedicated colour-bar slot
    cb = fig.colorbar(sc, cax=cax)
    cb.set_label(f"{rm['name']} vs {y0} %", color=MUTED)

    # Headline + subtitle (category note on its own line so nothing runs past the figure edge)
    if rev == "total":
        title = f"{BRAND_NAME[br]}: venues are sliding left ({xm['left']}) and up ({rm['up'][x]}) since {y0}"
        subtitle = (f"Same period Jan 1–Nov 8 each year · {xm['note']} · bubble size = revenue · "
                    f"dotted curves = constant revenue · colour = revenue vs {y0} (new venues vs their first year)")
    else:
        ex_all = g[g["cohort"] == "Existing"].groupby("year")["rev"].sum()
        chg = (ex_all[y1] / ex_all[y0] - 1) * 100
        k1 = int((g[(g["cohort"] == "Existing") & (g["year"] == y1)]["rev_vs_base"] < 0).sum())
        n1 = int(((g["cohort"] == "Existing") & (g["year"] == y1)).sum())
        title = (f"{BRAND_NAME[br]}: {rm['name'].replace(' revenue', '')} revenue at existing venues "
                 f"{chg:+.0f}% since {y0} — {k1} of {n1} venues below {y0}")
        subtitle = (f"Same period Jan 1–Nov 8 each year · {xm['note']} · bubble size = {rm['name'].lower()} · "
                    f"colour = vs {y0} (new venues vs first year)")
        fig.text(0.01, 0.958, rm["note2"], fontsize=11, color=MUTED, va="top")
    fig.text(0.01, 0.995, title, fontsize=16, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.975, subtitle, fontsize=11, color=MUTED, va="top")
    plt.show()

    # Supporting table: venue × year absolutes
    order = venue_order(br, y0, y1)
    rev_name = f"{rm['name']} ($M)"
    tbl = (g.assign(venue=pd.Categorical(g["venue"], categories=order, ordered=True))
             .sort_values(["venue", "year"])
             .assign(**{rev_name: (g["rev"] / 1e6).round(2)})
             .rename(columns={"venue": "Venue", "year": "Year", xc: xm["name"], "y": ylab,
                              "rev_vs_base": f"{rm['name']} vs {y0} %"})
             [["Venue", "Year", xm["name"], ylab, rev_name, f"{rm['name']} vs {y0} %"]]
             .round({xm["name"]: 0, ylab: 2, f"{rm['name']} vs {y0} %": 1}))
    tbl["Venue"] = tbl["Venue"].astype(str)
    display(tbl)

# Total
venue_abs_by_year("TRR", x="txn")
venue_abs_by_year("TRR", x="visits")
# Gaming
venue_abs_by_year("TRR", x="txn",    rev="games")
venue_abs_by_year("TRR", x="visits", rev="games")
# F&B
venue_abs_by_year("TRR", x="txn",    rev="fb")
venue_abs_by_year("TRR", x="visits", rev="fb")

In [0]:
# ============================================================
# Cell 11 — Revenue flow by venue: yearly F&B vs Gaming changes, 2023 → 2026 YTD
# Depends on: cohort_data, short, venue_order, BRAND_NAME, panel_title (Cell 10)
# ============================================================
FB_COLOR, GM_COLOR = "#3f8fb0", "#7d6fd6"

def flow_steps(fb_row, gm_row, years):
    """Consecutive-year (label, F&B Δ, Gaming Δ) steps, skipping years the venue wasn't open."""
    steps = []
    for ya, yb in zip(years[:-1], years[1:]):
        if pd.notna(fb_row.get(ya)) and pd.notna(fb_row.get(yb)):
            steps.append((f"→{str(yb)[2:]}", fb_row[yb] - fb_row[ya], gm_row[yb] - gm_row[ya]))
    return steps

def draw_flow(ax, steps, big=False):
    """Floating bars: each year's F&B then Gaming change, building on a running total from 0."""
    running, x, centers = 0.0, 0, []
    for _, fbd, gmd in steps:
        for val, c in [(fbd, FB_COLOR), (gmd, GM_COLOR)]:
            bottom = running if val >= 0 else running + val
            ax.bar(x, abs(val) / 1e6, bottom=bottom / 1e6, color=c, width=0.8, zorder=2)
            if big:
                ax.text(x, (bottom + abs(val)) / 1e6 if val >= 0 else bottom / 1e6,
                        f"{val/1e6:+.1f}", ha="center", va="bottom" if val >= 0 else "top",
                        fontsize=10, color=c, fontweight="bold")
            running += val
            ax.plot([x + 0.4, x + 0.6], [running / 1e6] * 2, color="#b4b2a9", lw=1, ls="--", zorder=1)
            x += 1
        centers.append(x - 1.5)
    ax.scatter(x - 0.4, running / 1e6, marker="D", s=60 if big else 40, color=INK, zorder=3)
    ax.annotate(f"{running/1e6:+.1f}M", (x - 0.4, running / 1e6), xytext=(8, 0),      # ← delete to hide end value
                textcoords="offset points", va="center", fontsize=12 if big else 10, color=INK, fontweight="bold")
    ax.axhline(0, color="#c3c2b7", lw=1, zorder=1)
    ax.set_xticks(centers, [s[0] for s in steps], fontsize=11 if big else 9)
    ax.set_xlim(-0.8, x + 1.4)
    ax.grid(axis="x", visible=False)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:+.1f}M" if v else "0"))
    return running

def revenue_flow(br, y0=2023, y1=2026):
    years = list(range(y0, y1 + 1))
    d = cohort_data(br)
    g = d[d["year"] >= y0].groupby(["Location_Name", "cohort", "year"])[["fb_revenue", "games_adj"]].sum()
    fb, gm = g["fb_revenue"].unstack("year"), g["games_adj"].unstack("year")
    order = venue_order(br, y0, y1)
    legend = [plt.Rectangle((0, 0), 1, 1, color=FB_COLOR, label="F&B change"),
              plt.Rectangle((0, 0), 1, 1, color=GM_COLOR, label="Gaming change"),
              Line2D([], [], marker="D", color=INK, ls="", label="Cumulative change since start")]

    # ---------- 1. All existing venues ----------
    ex_keys = [k for k in fb.index if k[1] == "Existing"]
    fb_ex, gm_ex = fb.loc[ex_keys].sum(), gm.loc[ex_keys].sum()
    steps_ex = flow_steps(fb_ex, gm_ex, years)

    fig, ax = plt.subplots(figsize=(14, 5.5))
    draw_flow(ax, steps_ex, big=True)
    ax.set_ylabel(f"Revenue change vs {y0} ($M)", color=MUTED, fontsize=11)
    ax.legend(handles=legend, loc="lower left", frameon=False, fontsize=10)
    fig.text(0.01, 0.995, f"{BRAND_NAME[br]} existing venues: revenue flow, {y0} → {y1} YTD",
             fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.94, "Year-by-year change, same period Jan 1–Nov 8 · each pair = one year's F&B then gaming "
             "change · gaming incl. unclassified lines during the 2023–24 coding gap · excludes kiosk sales",
             fontsize=11, color=MUTED, va="top")
    plt.tight_layout(rect=[0, 0, 1, 0.88]); plt.show()

    # ---------- 2. Per venue, shared $M scale (3 per row) ----------
    keys = sorted(fb.index, key=lambda k: order.index(short(pd.Series([k[0]]))[0] +
                                                      (" (new)" if k[1] == "New" else "")))
    ncol = 3
    nrow = int(np.ceil(len(keys) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(18, 4.6 * nrow), sharey=True)
    axes = np.atleast_2d(axes)
    rows = []
    for ax, (name, coh) in zip(axes.flat, keys):
        steps = flow_steps(fb.loc[(name, coh)], gm.loc[(name, coh)], years)
        draw_flow(ax, steps)
        nm = short(pd.Series([name]))[0] + (" (new)" if coh == "New" else "")
        panel_title(ax, nm)
        for lab, f_, g_ in steps:
            rows.append({"Venue": nm, "Step": f"{int('20' + lab[1:]) - 1}→20{lab[1:]}",
                         "F&B change ($M)": round(f_ / 1e6, 2), "Gaming change ($M)": round(g_ / 1e6, 2),
                         "Total change ($M)": round((f_ + g_) / 1e6, 2),
                         "Main driver": "F&B" if abs(f_) > abs(g_) else "Gaming"})
    for ax in axes.flat[len(keys):]:
        ax.set_visible(False)

    lo, hi = axes.flat[0].get_ylim()                              # headroom so end markers aren't clipped
    axes.flat[0].set_ylim(lo - 0.08 * (hi - lo), hi + 0.08 * (hi - lo))
    for ax in axes[:, 0]:
        ax.set_ylabel("Change vs start ($M)", color=MUTED, fontsize=10)
    fig.legend(handles=legend, loc="upper right", frameon=False, fontsize=10, ncol=3, bbox_to_anchor=(0.99, 0.985))
    fig.text(0.01, 0.995, f"{BRAND_NAME[br]}: revenue flow by venue — which part moved each year",
             fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.98, f"Same $M scale in every panel · existing venues from {y0}, new venues from their first "
             f"full year · sorted by cumulative transaction change (worst first)", fontsize=11, color=MUTED, va="top")
    plt.subplots_adjust(top=0.94, hspace=0.45, wspace=0.08); plt.show()

    display(pd.DataFrame(rows))

revenue_flow("TRR")

## Scene Visitation

In [0]:
# ============================================================
# Cell 12 — Scene+ visits & unique guests (read-only queries → pandas)
# Visit = member × date × venue, net qty > 0 (BR-013, BR-014)
# ============================================================
VISITS_CTE = """
WITH visits AS (
  SELECT LocationId, txn_date, CDE_ID
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v
  WHERE CDE_ID IS NOT NULL AND CDE_ID > 0
  GROUP BY LocationId, txn_date, CDE_ID
  HAVING SUM(Quantity) > 0
)
"""
# Daily visits per venue (sum-able to month / same-period year)
sv_daily = spark.sql(VISITS_CTE + """
SELECT LocationId, txn_date, COUNT(*) AS scene_visits
FROM visits GROUP BY LocationId, txn_date
""").toPandas()
sv_daily["txn_date"] = pd.to_datetime(sv_daily["txn_date"])
sv_daily["scene_visits"] = sv_daily["scene_visits"].astype(float)

# Unique guests (not sum-able across venues → brand level computed separately), same period Jan 1–Nov 8
df_sg = spark.sql(VISITS_CTE + """
SELECT l.LC_Trade_Brand_Description AS brand_name, v.LocationId, YEAR(v.txn_date) AS year,
       COUNT(*) AS scene_visits, COUNT(DISTINCT v.CDE_ID) AS scene_guests
FROM visits v
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON v.LocationId = l.LocationID
WHERE DATE_FORMAT(v.txn_date, 'MM-dd') <= '11-08'
GROUP BY GROUPING SETS ((l.LC_Trade_Brand_Description, YEAR(v.txn_date)),
                        (l.LC_Trade_Brand_Description, v.LocationId, YEAR(v.txn_date)))
""").toPandas()
df_sg[["scene_visits", "scene_guests"]] = df_sg[["scene_visits", "scene_guests"]].astype(float)
df_sg["LocationId"] = df_sg["LocationId"].astype("Int64")      # NULL = brand-level row
df_sg = df_sg.merge(venue[["LocationId", "Location_Name"]].astype({"LocationId": "Int64"}),
                    on="LocationId", how="left")
print(sv_daily.shape, df_sg.shape)

In [0]:
# ============================================================
# Cell 13 — Scene+ analysis (standalone; Cells 10–11 untouched)
# (1) trend · (2) venue views · (3) revenue flow · (+) guests
# Depends on: Cell 7 (trend_panel, BLUE, ORANGE), Cell 10 (BRAND_NAME, PROXY_BRAND, BREAK_WINDOWS, short,
#             venue_order, panel_title, REV_MODES), Cell 11 (flow_steps, draw_flow, FB_COLOR, GM_COLOR),
#             Cell 12 (sv_daily, df_sg)
# ============================================================
SCENE_NOTE = "Scene+ growth partly reflects more guests scanning their card, not only more demand"
SCENE_X = {
    "txn":    dict(col="n_txn",  label="Scene+ transactions (Jan 1–Nov 8)", name="Scene+ transactions",
                   note="Scene+ transaction = any line with a member ID"),
    "visits": dict(col="visits", label="Scene+ visits (Jan 1–Nov 8)",       name="Scene+ visits",
                   note="Scene+ visit = member × date × venue (actual count)"),
}

def scene_cohort_data(br):
    """Same-period (Jan 1–Nov 8) Scene+ rows for one brand, with cohort and gap-adjusted gaming."""
    d = (df_base[(df_base["brand"] == br) & df_base["in_ytd_window"] & (df_base["is_scene"] == 1)]
         .merge(venue[["LocationId", "cohort"]], on="LocationId"))
    s, e = BREAK_WINDOWS[br]
    unclassified = d["revenue"] - d["fb_revenue"] - d["games_revenue"]
    d["games_adj"] = d["games_revenue"] + np.where(d["txn_date"].between(s, e), unclassified, 0)
    return d

# ---------- (1) Trend: Scene+ visits and spend per visit — LBE, TRR, PDM ----------
def scene_trend():
    v = sv_daily.merge(venue[["LocationId", "brand"]], on="LocationId")
    v["month"] = v["txn_date"].dt.to_period("M").dt.to_timestamp()
    cutoff = pd.Timestamp("2026-11-01")                                         # drop partial Nov 2026
    vm = v[v["month"] < cutoff].groupby(["brand", "month"])["scene_visits"].sum()
    rm_ = df_base[df_base["month"] < cutoff].groupby(["brand", "month"])["rev_scene"].sum()
    m = pd.concat([vm, rm_], axis=1).reset_index()
    series = {"LBE": m.groupby("month")[["scene_visits", "rev_scene"]].sum(),
              "TRR": m[m["brand"] == "TRR"].set_index("month")[["scene_visits", "rev_scene"]],
              "PDM": m[m["brand"] == "PDM"].set_index("month")[["scene_visits", "rev_scene"]]}
    titles = {"LBE": "All LBE", "TRR": "The Rec Room", "PDM": "Playdium"}

    fig, axes = plt.subplots(2, 3, figsize=(24, 9), sharex=True)
    for col, key in enumerate(["LBE", "TRR", "PDM"]):
        s = series[key].sort_index()
        rv, rr = s["scene_visits"].rolling(12, center=True), s["rev_scene"].rolling(12, center=True)
        trend_panel(axes[0, col], s["scene_visits"], rv.mean(), BLUE, "Scene+ visits",
                    lambda x: f"{x/1e3:.0f}K", lambda x: f"{x/1e3:.0f}K", f"{titles[key]} — Scene+ visits per month")
        trend_panel(axes[1, col], s["rev_scene"] / s["scene_visits"], rr.sum() / rv.sum(), ORANGE,
                    "$ per Scene+ visit", lambda x: f"${x:.0f}", lambda x: f"${x:.2f}",
                    f"{titles[key]} — Scene+ spend per visit")
    for ax in axes.flat:
        if ax not in (axes[0, 0], axes[1, 0]):
            ax.get_legend().remove()
    for ax in axes[1]:
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.set_xlim(m["month"].min() - pd.Timedelta(days=20), m["month"].max() + pd.Timedelta(days=200))

    # Headline from same-period numbers (Jan 1–Nov 8, 2023 vs 2026), consistent with all other views
    bvis = df_sg[df_sg["LocationId"].isna()].copy()
    bvis["brand"] = bvis["brand_name"].map(BRAND_SHORT)
    vis = bvis.pivot_table(index="year", columns="brand", values="scene_visits", aggfunc="sum")
    vis["LBE"] = vis.sum(axis=1)
    rev = df_base[df_base["in_ytd_window"]].pivot_table(index="year", columns="brand", values="rev_scene", aggfunc="sum")
    rev["LBE"] = rev.sum(axis=1)
    pct = lambda s: (s[2026] / s[2023] - 1) * 100
    v_pct = {k: pct(vis[k]) for k in ["LBE", "TRR", "PDM"]}
    s_pct = {k: pct(rev[k] / vis[k]) for k in ["LBE", "TRR", "PDM"]}
    fig.text(0.01, 0.995, f"Scene+ 2023 → 2026 YTD — visits: LBE {v_pct['LBE']:+.0f}%, Rec Room {v_pct['TRR']:+.0f}%, "
             f"Playdium {v_pct['PDM']:+.0f}% · spend per visit: LBE {s_pct['LBE']:+.0f}%",
             fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.955, "Headline = same period Jan 1–Nov 8, all venues · lines = monthly actual and centred "
             f"12-month trend (labels on lines show trend-line change) · {SCENE_NOTE}",
             fontsize=11, color=MUTED, va="top")
    plt.tight_layout(rect=[0, 0, 1, 0.92]); plt.show()

# ---------- (2) Venue views: 2×2 by year ----------
def scene_abs_by_year(br, x="visits", rev="total", y0=2023, y1=2026):
    xm, rm = SCENE_X[x], REV_MODES[rev]
    xc, ylab = xm["col"], "Scene+ " + rm["y"][x]
    vlim = 60                                                      # Scene+ growth exceeds ±30%
    d = scene_cohort_data(br)
    g = (d[d["year"] >= y0].groupby(["Location_Name", "cohort", "year"])
           [["n_txn", "revenue", "games_adj", "fb_revenue"]].sum().reset_index())
    sv = df_sg[df_sg["LocationId"].notna()][["Location_Name", "year", "scene_visits"]]
    g = g.merge(sv, on=["Location_Name", "year"], how="left").rename(columns={"scene_visits": "visits"})
    g["rev"]   = g[rm["col"]]
    g["y"]     = g["rev"] / g[xc]
    g["venue"] = short(g["Location_Name"]) + np.where(g["cohort"] == "New", " (new)", "")
    first_rev = g.sort_values("year").groupby("Location_Name")["rev"].transform("first")
    g["rev_vs_base"] = (g["rev"] / first_rev - 1) * 100

    xr = (0, g[xc].max() * 1.15)
    yr = (g["y"].min() * 0.9, g["y"].max() * 1.08)
    smax = g["rev"].max()
    step = 1e6 if smax < 6e6 else 2e6
    levels = np.arange(step, smax * 1.3, step)
    xs = np.linspace(xr[1] * 0.02, xr[1], 400)

    years = list(range(y0, y1 + 1))
    fig, axes = plt.subplots(2, 2, figsize=(18, 15), sharex=True, sharey=True)
    for ax, yr_i in zip(axes.flat, years):
        h = g[g["year"] == yr_i]
        for lv in levels:
            ys = lv / xs
            ok = (ys >= yr[0]) & (ys <= yr[1])
            ax.plot(xs[ok], ys[ok], color="#c3c2b7", lw=1, ls=":", zorder=0)
            if ok.any():
                ax.text(xs[ok][-1], ys[ok][-1], f"${lv/1e6:.0f}M ", fontsize=9, color=MUTED, ha="right", va="bottom")
        for mk, sub in [("o", h["cohort"] == "Existing"), ("D", h["cohort"] == "New")]:
            if sub.any():
                sc = ax.scatter(h.loc[sub, xc], h.loc[sub, "y"], s=1800 * h.loc[sub, "rev"] / smax + 80, marker=mk,
                                c=h.loc[sub, "rev_vs_base"], cmap=rm["cmap"], vmin=-vlim, vmax=vlim,
                                edgecolor="white", linewidth=1.2, alpha=0.9, zorder=3)
        for _, row in h.iterrows():
            ax.annotate(row["venue"], (row[xc], row["y"]), xytext=(9, -3),
                        textcoords="offset points", fontsize=9, color=INK, va="center")
        ex = h[h["cohort"] == "Existing"]
        k = int((ex["rev_vs_base"] < 0).sum())
        sub_txt = "baseline" if yr_i == y0 else f"{k} of {len(ex)} existing venues below {y0}"
        panel_title(ax, f"{yr_i}{' YTD' if yr_i == y1 else ''}: {rm['short']} ${h['rev'].sum()/1e6:,.1f}M · {sub_txt}")
        ax.set_xlim(*xr); ax.set_ylim(*yr)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1e3:,.0f}K"))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:.0f}"))

    for ax in axes[1]:
        ax.set_xlabel(xm["label"], color=MUTED, fontsize=11)
    for ax in axes[:, 0]:
        ax.set_ylabel(ylab, color=MUTED, fontsize=11)
    plt.subplots_adjust(top=0.92, right=0.90, hspace=0.18, wspace=0.06)
    cax = fig.add_axes([0.92, 0.30, 0.012, 0.40])
    cb = fig.colorbar(sc, cax=cax)
    cb.set_label(f"Scene+ {rm['name'].lower()} vs {y0} %", color=MUTED)

    ex_all = g[g["cohort"] == "Existing"].groupby("year")["rev"].sum()
    chg = (ex_all[y1] / ex_all[y0] - 1) * 100
    last = g[(g["cohort"] == "Existing") & (g["year"] == y1)]
    cat = "revenue" if rev == "total" else rm["name"].lower()
    fig.text(0.01, 0.995, f"{BRAND_NAME[br]} Scene+: {cat} at existing venues {chg:+.0f}% since {y0} — "
             f"{int((last['rev_vs_base'] < 0).sum())} of {len(last)} venues below {y0}",
             fontsize=16, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.975, f"Same period Jan 1–Nov 8 each year · {xm['note']} · bubble size = Scene+ {cat} · "
             f"dotted curves = constant Scene+ {cat} · colour = vs {y0} (new venues vs first year)",
             fontsize=11, color=MUTED, va="top")
    fig.text(0.01, 0.958, " · ".join(n for n in [rm.get("note2"), SCENE_NOTE] if n),
             fontsize=11, color=MUTED, va="top")
    plt.show()

    order = venue_order(br, y0, y1)
    rev_name, vs_name = f"Scene+ {rm['name']} ($M)", f"Scene+ {rm['name']} vs {y0} %"
    tbl = (g.assign(venue=pd.Categorical(g["venue"], categories=order, ordered=True))
             .sort_values(["venue", "year"])
             .assign(**{rev_name: (g["rev"] / 1e6).round(2)})
             .rename(columns={"venue": "Venue", "year": "Year", xc: xm["name"], "y": ylab, "rev_vs_base": vs_name})
             [["Venue", "Year", xm["name"], ylab, rev_name, vs_name]]
             .round({xm["name"]: 0, ylab: 2, vs_name: 1}))
    tbl["Venue"] = tbl["Venue"].astype(str)
    display(tbl)

# ---------- (3) Revenue flow: yearly Scene+ F&B vs gaming changes ----------
def scene_revenue_flow(br, y0=2023, y1=2026):
    years = list(range(y0, y1 + 1))
    d = scene_cohort_data(br)
    g = d[d["year"] >= y0].groupby(["Location_Name", "cohort", "year"])[["fb_revenue", "games_adj"]].sum()
    fb, gm = g["fb_revenue"].unstack("year"), g["games_adj"].unstack("year")
    order = venue_order(br, y0, y1)
    legend = [plt.Rectangle((0, 0), 1, 1, color=FB_COLOR, label="Scene+ F&B change"),
              plt.Rectangle((0, 0), 1, 1, color=GM_COLOR, label="Scene+ gaming change"),
              Line2D([], [], marker="D", color=INK, ls="", label="Cumulative change since start")]

    ex_keys = [k for k in fb.index if k[1] == "Existing"]
    fig, ax = plt.subplots(figsize=(14, 5.5))
    draw_flow(ax, flow_steps(fb.loc[ex_keys].sum(), gm.loc[ex_keys].sum(), years), big=True)
    ax.set_ylabel(f"Scene+ revenue change vs {y0} ($M)", color=MUTED, fontsize=11)
    ax.legend(handles=legend, loc="upper left", frameon=False, fontsize=10)
    fig.text(0.01, 0.995, f"{BRAND_NAME[br]} existing venues: Scene+ revenue flow, {y0} → {y1} YTD",
             fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.94, "Year-by-year change, same period Jan 1–Nov 8 · gaming incl. unclassified lines during "
             f"the 2023–24 coding gap · excludes kiosk sales · {SCENE_NOTE}", fontsize=11, color=MUTED, va="top")
    plt.tight_layout(rect=[0, 0, 1, 0.88]); plt.show()

    keys = sorted(fb.index, key=lambda k: order.index(short(pd.Series([k[0]]))[0] +
                                                      (" (new)" if k[1] == "New" else "")))
    nrow = int(np.ceil(len(keys) / 3))
    fig, axes = plt.subplots(nrow, 3, figsize=(18, 4.6 * nrow), sharey=True)
    axes = np.atleast_2d(axes)
    rows = []
    for ax, (name, coh) in zip(axes.flat, keys):
        steps = flow_steps(fb.loc[(name, coh)], gm.loc[(name, coh)], years)
        draw_flow(ax, steps)
        nm = short(pd.Series([name]))[0] + (" (new)" if coh == "New" else "")
        panel_title(ax, nm)
        for lab, f_, g_ in steps:
            rows.append({"Venue": nm, "Step": f"{int('20' + lab[1:]) - 1}→20{lab[1:]}",
                         "Scene+ F&B change ($M)": round(f_ / 1e6, 2), "Scene+ gaming change ($M)": round(g_ / 1e6, 2),
                         "Scene+ total change ($M)": round((f_ + g_) / 1e6, 2),
                         "Main driver": "F&B" if abs(f_) > abs(g_) else "Gaming"})
    for ax in axes.flat[len(keys):]:
        ax.set_visible(False)
    lo, hi = axes.flat[0].get_ylim()
    axes.flat[0].set_ylim(lo - 0.08 * (hi - lo), hi + 0.08 * (hi - lo))
    for ax in axes[:, 0]:
        ax.set_ylabel("Change vs start ($M)", color=MUTED, fontsize=10)
    fig.legend(handles=legend, loc="upper right", frameon=False, fontsize=10, ncol=3, bbox_to_anchor=(0.99, 0.985))
    fig.text(0.01, 0.995, f"{BRAND_NAME[br]}: Scene+ revenue flow by venue — which part moved each year",
             fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.98, f"Same $M scale in every panel · existing venues from {y0}, new venues from their first "
             "full year · sorted by cumulative transaction change (worst first)", fontsize=11, color=MUTED, va="top")
    plt.subplots_adjust(top=0.94, hspace=0.45, wspace=0.08); plt.show()
    display(pd.DataFrame(rows))

# ---------- (+) Guests: more members, or more frequent members? ----------
def scene_guests(br, y0=2023):
    bn = PROXY_BRAND[br]
    s = df_sg[(df_sg["brand_name"] == bn) & (df_sg["year"] >= y0)].copy()
    s["Venue"] = np.where(s["LocationId"].isna(), f"All {BRAND_NAME[br]} (unique)",
                          short(s["Location_Name"].fillna("")))
    s["Visits per guest"] = (s["scene_visits"] / s["scene_guests"]).round(2)
    base = s.sort_values("year").groupby("Venue")[["scene_guests", "scene_visits"]].transform("first")
    s["Guests vs first yr %"] = (100 * (s["scene_guests"] / base["scene_guests"] - 1)).round(1)
    s["Visits vs first yr %"] = (100 * (s["scene_visits"] / base["scene_visits"] - 1)).round(1)
    order = [f"All {BRAND_NAME[br]} (unique)"] + [v.replace(" (new)", "") for v in venue_order(br)]
    tbl = (s.assign(Venue=pd.Categorical(s["Venue"], categories=order, ordered=True))
             .sort_values(["Venue", "year"])
             .rename(columns={"year": "Year", "scene_guests": "Unique guests", "scene_visits": "Scene+ visits"})
             [["Venue", "Year", "Unique guests", "Scene+ visits", "Visits per guest",
               "Guests vs first yr %", "Visits vs first yr %"]])
    tbl["Venue"] = tbl["Venue"].astype(str)
    display(tbl)

# ---------- Run ----------
scene_trend()                                                     # (1)
for x_ in ["txn", "visits"]:                                      # (2)
    for rev_ in ["total", "games", "fb"]:
        scene_abs_by_year("TRR", x=x_, rev=rev_)
scene_revenue_flow("TRR")                                         # (3)
scene_guests("TRR")                                               # (+)

In [0]:
# ============================================================
# Cell 14 — Scene+ new vs returning guests by venue
# New = first visit to ANY LBE venue (TRR or PDM) since 2022-01-01 falls in this year
# Returning = visited any LBE venue in an earlier year (since 2022)
# Visit = member × date × venue, net qty > 0 · visit revenue = all transactions that member made there that day
# ============================================================
NR_SQL = """
WITH id_lines AS (
  SELECT LocationId, txn_date, Transaction_Number, CDE_ID, Quantity
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v
  WHERE CDE_ID IS NOT NULL AND CDE_ID > 0 AND txn_date >= '2022-01-01'
),
visits AS (
  SELECT LocationId, txn_date, CDE_ID FROM id_lines
  GROUP BY LocationId, txn_date, CDE_ID HAVING SUM(Quantity) > 0
),
first_lbe_visit AS (                                   -- first visit to ANY LBE venue since 2022
  SELECT CDE_ID, MIN(txn_date) AS first_date FROM visits GROUP BY CDE_ID
),
txn_rev AS (
  SELECT LocationId, txn_date, Transaction_Number, SUM(Net_Revenue) AS rev, SUM(Quantity) AS q
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v
  WHERE txn_date >= '2023-01-01'
  GROUP BY LocationId, txn_date, Transaction_Number
),
txn_cde AS (
  SELECT DISTINCT LocationId, txn_date, Transaction_Number, CDE_ID
  FROM id_lines WHERE txn_date >= '2023-01-01'
),
visit_txn AS (
  SELECT v.LocationId, v.txn_date, v.CDE_ID,
         SUM(CASE WHEN t.q > 0 THEN 1 ELSE 0 END) AS n_txn,
         SUM(t.rev)                               AS revenue
  FROM visits v
  JOIN txn_cde c ON c.LocationId = v.LocationId AND c.txn_date = v.txn_date AND c.CDE_ID = v.CDE_ID
  JOIN txn_rev t ON t.LocationId = c.LocationId AND t.txn_date = c.txn_date
                AND t.Transaction_Number = c.Transaction_Number
  WHERE v.txn_date >= '2023-01-01'
  GROUP BY v.LocationId, v.txn_date, v.CDE_ID
)
SELECT vt.LocationId, YEAR(vt.txn_date) AS year,
       CASE WHEN YEAR(f.first_date) = YEAR(vt.txn_date) THEN 'New to LBE' ELSE 'Returning' END AS cohort,
       COUNT(DISTINCT vt.CDE_ID) AS guests,
       COUNT(*)                  AS visits,
       SUM(vt.n_txn)             AS n_txn,
       SUM(vt.revenue)           AS revenue
FROM visit_txn vt
JOIN first_lbe_visit f ON f.CDE_ID = vt.CDE_ID
WHERE DATE_FORMAT(vt.txn_date, 'MM-dd') <= '11-08'
GROUP BY vt.LocationId, YEAR(vt.txn_date),
         CASE WHEN YEAR(f.first_date) = YEAR(vt.txn_date) THEN 'New to LBE' ELSE 'Returning' END
"""
df_nr = spark.sql(NR_SQL).toPandas()
df_nr[["guests", "visits", "n_txn", "revenue"]] = df_nr[["guests", "visits", "n_txn", "revenue"]].astype(float)
df_nr = df_nr.merge(venue[["LocationId", "brand", "Location_Name", "cohort"]].rename(columns={"cohort": "venue_cohort"}),
                    on="LocationId")
df_nr["Venue"] = short(df_nr["Location_Name"]) + np.where(df_nr["venue_cohort"] == "New", " (new)", "")

In [0]:
def new_vs_returning(br):
    d = df_nr[df_nr["brand"] == br]
    COH = ["New to LBE", "Returning"]
    COL = {"New to LBE": "#1baf7a", "Returning": BLUE}

    # ---- Brand-level chart (guest-venue counts: a guest at two venues counts at each) ----
    b = d.groupby(["year", "cohort"])[["guests", "visits", "n_txn", "revenue"]].sum().reset_index()
    b["spend_per_visit"] = b["revenue"] / b["visits"]
    b["visits_per_guest"] = b["visits"] / b["guests"]
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    piv = b.pivot(index="year", columns="cohort", values="guests").reindex(columns=COH).fillna(0)
    bottom = np.zeros(len(piv))
    for c in COH:
        axes[0].bar(piv.index.astype(str), piv[c] / 1e3, bottom=bottom / 1e3, color=COL[c], label=c, width=0.6)
        bottom += piv[c].values
    for i, (yr_, row) in enumerate(piv.iterrows()):
        share = 100 * row[COH[0]] / row.sum()
        axes[0].text(i, row.sum() / 1e3, f"{share:.0f}% new", ha="center", va="bottom", fontsize=11, color=INK)
    panel_title(axes[0], "Guests by cohort")
    axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}K"))
    axes[0].legend(frameon=False, fontsize=10)
    for ax, col, ttl, fmt in [(axes[1], "visits_per_guest", "Visits per guest", lambda v: f"{v:.2f}"),
                              (axes[2], "spend_per_visit", "Spend per visit ($)", lambda v: f"${v:.0f}")]:
        for c in COH:
            s = b[b["cohort"] == c].set_index("year")[col]
            ax.plot(s.index.astype(str), s, marker="o", lw=2.8, color=COL[c], label=c)
            for x_, y_ in s.items():
                ax.annotate(fmt(y_), (str(x_), y_), xytext=(0, 8), textcoords="offset points",
                            ha="center", fontsize=10, color=COL[c])
        panel_title(ax, ttl)
    fig.text(0.01, 0.995, f"{BRAND_NAME[br]} Scene+: new vs returning guests, 2023 → 2026 YTD",
             fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.94, "Same period Jan 1–Nov 8 · New = first visit to any LBE venue (Rec Room or Playdium) "
             "since Jan 2022 · Returning = visited any LBE venue in an earlier year · 'New' includes guests lapsed "
             "before 2022 · guests counted at each venue they visit", fontsize=11, color=MUTED, va="top")
    plt.tight_layout(rect=[0, 0, 1, 0.88]); plt.show()

    # ---- Venue table ----
    t = d.groupby(["Venue", "year", "cohort"])[["guests", "visits", "n_txn", "revenue"]].sum().reset_index()
    tot = t.groupby(["Venue", "year"])["guests"].transform("sum")
    t["Guest share %"]       = (100 * t["guests"] / tot).round(1)
    t["Visits per guest"]    = (t["visits"] / t["guests"]).round(2)
    t["Txns per visit"]      = (t["n_txn"] / t["visits"]).round(2)
    t["Spend per visit ($)"] = (t["revenue"] / t["visits"]).round(2)
    t["Yield ($ per txn)"]   = (t["revenue"] / t["n_txn"]).round(2)
    order = venue_order(br)
    t = (t.assign(Venue=pd.Categorical(t["Venue"], categories=order, ordered=True))
          .sort_values(["Venue", "year", "cohort"])
          .rename(columns={"year": "Year", "cohort": "Cohort", "guests": "Guests", "visits": "Visits"})
          [["Venue", "Year", "Cohort", "Guests", "Guest share %", "Visits", "Visits per guest",
            "Txns per visit", "Spend per visit ($)", "Yield ($ per txn)"]])
    t["Venue"] = t["Venue"].astype(str)
    display(t)

new_vs_returning("TRR")

In [0]:
# ============================================================
# Cell 15 — Game Band sales mix: original purchases vs reloads, same period Jan 1–Nov 8, 2023–2026
# Tier + credits parsed from item name · bins merged: $10+$15, $100+$129 (Summer Pass analysis rule)
# ============================================================
import re

BAND_SQL = """
SELECT l.LC_Trade_Brand_Description AS brand_name, v.LocationId, YEAR(v.txn_date) AS year,
       i.Item_Description,
       CASE WHEN v.CDE_ID IS NOT NULL AND v.CDE_ID > 0 THEN 1 ELSE 0 END AS is_scene,
       SUM(v.Quantity)    AS units,
       SUM(v.Net_Revenue) AS revenue
FROM data_exploration_hub.marketing_hub.lbe_review_lines_v v
JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i     ON v.ItemSkey = i.itemskey
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON v.LocationId = l.LocationID
WHERE v.category = 'Games'
  AND UPPER(i.Item_Description) LIKE '%TAG%'
  AND YEAR(v.txn_date) >= 2023
  AND DATE_FORMAT(v.txn_date, 'MM-dd') <= '11-08'
GROUP BY 1, 2, 3, 4, 5
"""
df_band = spark.sql(BAND_SQL).toPandas()
df_band[["units", "revenue"]] = df_band[["units", "revenue"]].astype(float)
df_band["brand"] = df_band["brand_name"].map(BRAND_SHORT)

# ---- Parse name: tier ($), credits, band type; drop free-game items ----
desc = df_band["Item_Description"].str.upper()
df_band = df_band[~desc.str.contains("FREE")].copy()
desc = df_band["Item_Description"].str.upper()
df_band["tier_usd"]  = desc.str.extract(r"\$\s*(\d+)")[0].astype(float)
df_band["credits"]   = desc.str.extract(r"(\d+)\s*CR")[0].astype(float)
df_band["band_type"] = np.select([desc.str.contains("PROMO"), desc.str.startswith("BP"), desc.str.startswith("PREM")],
                                 ["Promo", "BP", "Premium"], default="Standard")

# ---- Purchase vs reload (CONFIRM: edit here if any mapping is wrong) ----
PURCHASE_MAP = {"BP": "Band purchase", "Premium": "Band purchase", "Promo": "Band purchase", "Standard": "Reload"}
df_band["purchase"] = df_band["band_type"].map(PURCHASE_MAP)

# ---- Merged bins ($10+$15, $100+$129), ordered numerically ----
BIN_MERGE = {15: 10, 129: 100}
BIN_LABEL = {10: "$10–15", 100: "$100–129"}
df_band["bin_usd"] = df_band["tier_usd"].replace(BIN_MERGE)
df_band["tier"] = df_band["bin_usd"].map(lambda v: BIN_LABEL.get(v, f"${int(v)}") if pd.notna(v) else None)

# Monthly band units — to see the coding-gap months (read-only)
BAND_MONTHLY_SQL = """
SELECT l.LC_Trade_Brand_Description AS brand_name, DATE_TRUNC('MONTH', v.txn_date) AS month,
       SUM(v.Quantity) AS units
FROM data_exploration_hub.marketing_hub.lbe_review_lines_v v
JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i     ON v.ItemSkey = i.itemskey
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON v.LocationId = l.LocationID
WHERE v.category = 'Games' AND UPPER(i.Item_Description) LIKE '%TAG%'
  AND UPPER(i.Item_Description) NOT LIKE '%FREE%' AND v.txn_date >= '2023-01-01'
GROUP BY 1, 2
"""
df_band_m = spark.sql(BAND_MONTHLY_SQL).toPandas()
df_band_m["units"] = df_band_m["units"].astype(float)
df_band_m["brand"] = df_band_m["brand_name"].map(BRAND_SHORT)

# ---- Parse name: tier ($, decimals allowed), credits, band type; drop free-game items ----
desc = df_band["Item_Description"].str.upper()
df_band = df_band[~desc.str.contains("FREE") & ~desc.str.startswith("G/E")].copy()   # G/E also excluded in SQL
desc = df_band["Item_Description"].str.upper()
df_band["tier_usd"]  = desc.str.extract(r"\$\s*(\d+(?:\.\d+)?)")[0].astype(float)
df_band["credits"]   = desc.str.extract(r"(\d+)\s*CR")[0].astype(float)
df_band["band_type"] = np.select([desc.str.contains("BOGO"), desc.str.contains("PROMO"),
                                  desc.str.contains(r"\bBP\b"), desc.str.startswith("PREM")],
                                 ["BOGO", "Promo", "BP", "Premium"], default="Standard")

# ---- Purchase vs reload (confirmed: BP = load + $2.50 band fee, PREM = load + $5.50, TAG = reload) ----
PURCHASE_MAP = {"BP": "Band purchase", "Premium": "Band purchase", "Promo": "Band purchase",
                "Standard": "Reload", "BOGO": "Promo (excluded)"}
df_band["purchase"] = df_band["band_type"].map(PURCHASE_MAP)



In [0]:
# ============================================================
# Chart + table: mix for one brand and one purchase type
# ============================================================
def band_mix(br, purchase="Band purchase"):
    d = df_band[(df_band["brand"] == br) & (df_band["purchase"] == purchase) & df_band["bin_usd"].notna()].copy()
    order = d.drop_duplicates("tier").sort_values("bin_usd")["tier"].tolist()
    pal = sns.color_palette("crest", len(order))
    t = d.groupby(["year", "tier"]).agg(units=("units", "sum"), revenue=("revenue", "sum")).reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(24, 6.5))
    for ax, col, ttl in [(axes[0], "units", "Share of units"), (axes[1], "revenue", "Share of revenue")]:
        piv = t.pivot(index="year", columns="tier", values=col).reindex(columns=order).fillna(0)
        piv = 100 * piv.div(piv.sum(axis=1), axis=0)
        bottom = np.zeros(len(piv))
        for c, color in zip(order, pal):
            ax.bar(piv.index.astype(str), piv[c], bottom=bottom, color=color, width=0.65, label=c)
            for i, (v_, b_) in enumerate(zip(piv[c], bottom)):
                if v_ >= 6:
                    ax.text(i, b_ + v_ / 2, f"{v_:.0f}%", ha="center", va="center", fontsize=9, color="white")
            bottom += piv[c].values
        ax.set_ylim(0, 100); ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}%"))
        panel_title(ax, ttl)
    axes[1].legend(title="Tier", frameon=False, fontsize=9, bbox_to_anchor=(1.0, 1.0), loc="upper left")

    yr = d.groupby("year").agg(units=("units", "sum"), revenue=("revenue", "sum"))
    avg = yr["revenue"] / yr["units"]
    axes[2].plot(avg.index.astype(str), avg, marker="o", lw=2.8, color=GM_COLOR)
    for x_, y_ in avg.items():
        axes[2].annotate(f"${y_:.2f}", (str(x_), y_), xytext=(0, 8), textcoords="offset points",
                         ha="center", fontsize=10, color=GM_COLOR)
    axes[2].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:.0f}"))
    panel_title(axes[2], f"Average $ per {purchase.lower()}")

    label = "original band purchases" if purchase == "Band purchase" else "reloads"
    fig.text(0.01, 0.995, f"{BRAND_NAME[br]}: Game Band mix — {label}, 2023 → 2026 YTD",
             fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.94, r"Same period Jan 1–Nov 8 · bins merged \$10+\$15 and \$100+\$129 · free-game items excluded · "
             "2023–24 counts understated by the coding gap (shares less affected) · excludes kiosk sales",
             fontsize=11, color=MUTED, va="top")
    plt.tight_layout(rect=[0, 0, 1, 0.88]); plt.show()

    # ---- Table: tier × year ----
    d["cr_per_usd_u"] = d["units"] * d["credits"] / d["tier_usd"]
    d["sc_units"]     = d["units"] * d["is_scene"]
    tb = (d.groupby(["tier", "bin_usd", "year"])
           .agg(units=("units", "sum"), revenue=("revenue", "sum"),
                cr=("cr_per_usd_u", "sum"), sc=("sc_units", "sum")).reset_index())
    tb["Unit share %"]    = (100 * tb["units"] / tb.groupby("year")["units"].transform("sum")).round(1)
    tb["Revenue share %"] = (100 * tb["revenue"] / tb.groupby("year")["revenue"].transform("sum")).round(1)
    tb["Avg $ per unit"]  = (tb["revenue"] / tb["units"]).round(2)
    tb["Credits per $"]   = (tb["cr"] / tb["units"]).round(2)
    tb["Scene+ share %"]  = (100 * tb["sc"] / tb["units"]).round(1)
    tb = (tb.sort_values(["bin_usd", "year"])
            .rename(columns={"tier": "Tier", "year": "Year", "units": "Units"})
            [["Tier", "Year", "Units", "Unit share %", "Revenue share %", "Avg $ per unit",
              "Credits per $", "Scene+ share %"]])
    display(tb)

band_mix("TRR", "Band purchase")
band_mix("TRR", "Reload")

Scene Attachment

In [0]:

a = views[("Year_SamePeriod", "Brand")].sort_values(["brand", "year"]).copy()
a["Attachment % (txns)"]   = (100 * a["scene_txn_share"]).round(1)
a["Attachment % (revenue)"] = (100 * a["scene_rev_share"]).round(1)
base23 = a[a["year"] == 2023].set_index("brand")["scene_txn_share"]
a["Scene+ txns if attachment flat at 2023"] = (a["n_txn"] * a["brand"].map(base23)).round(0)
print(a[["brand", "year", "n_txn", "n_txn_scene", "Attachment % (txns)", "Attachment % (revenue)",
         "Scene+ txns if attachment flat at 2023"]].to_string(index=False))

Scene Visitation Frequency

In [0]:
freq = spark.sql("""
WITH visits AS (
  SELECT v.LocationId, v.txn_date, v.CDE_ID
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v v
  WHERE v.CDE_ID IS NOT NULL AND v.CDE_ID > 0
    AND YEAR(v.txn_date) >= 2023 AND DATE_FORMAT(v.txn_date, 'MM-dd') <= '11-08'
  GROUP BY v.LocationId, v.txn_date, v.CDE_ID
  HAVING SUM(v.Quantity) > 0
),
per_member AS (
  SELECT l.LC_Trade_Brand_Description AS brand, YEAR(v.txn_date) AS yr, v.CDE_ID, COUNT(*) AS n_visits
  FROM visits v JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON v.LocationId = l.LocationID
  GROUP BY 1, 2, 3
)
SELECT brand, yr,
       CASE WHEN n_visits = 1 THEN '1' WHEN n_visits <= 3 THEN '2-3' WHEN n_visits <= 6 THEN '4-6'
            WHEN n_visits <= 12 THEN '7-12' ELSE '13+' END AS visit_bin,
       COUNT(*) AS members, SUM(n_visits) AS visits
FROM per_member GROUP BY 1, 2, 3
""").toPandas()
freq[["members", "visits"]] = freq[["members", "visits"]].astype(float)

tot = freq.groupby(["brand", "yr"])[["members", "visits"]].sum()
print("Average visits per member:\n", (tot["visits"] / tot["members"]).round(2).unstack("yr").to_string())
share = (100 * freq.pivot_table(index=["brand", "yr"], columns="visit_bin", values="members", aggfunc="sum")
         .reindex(columns=["1", "2-3", "4-6", "7-12", "13+"])
         .pipe(lambda x: x.div(x.sum(axis=1), axis=0))).round(1)
print("\n% of members by visits per year:\n", share.to_string())

Scene Gameband & Reload Pattern

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

BR_COL  = {"TRR": BLUE, "PDM": ORANGE, "Rec Room": BLUE, "Playdium": ORANGE}
BR_FULL = {"TRR": "Rec Room", "PDM": "Playdium"}
YR_LBL  = {2022: "2022", 2023: "2023", 2024: "2024", 2025: "2025", 2026: "2026 YTD"}
PERIOD  = "Same period each year: Jan 1 – Nov 8"
kfmt    = mtick.FuncFormatter(lambda v, _: f"{v/1e3:,.0f}K")

def header(fig, title, sub):
    fig.text(0.01, 0.98, title, fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 0.92, sub, fontsize=10.5, color=MUTED, va="top")

# ---------- Q1a: attachment rate by year ----------
fig, ax = plt.subplots(figsize=(10, 5.2))
for br in ["TRR", "PDM"]:
    d = a[a["brand"] == br].sort_values("year")
    ax.plot(d["year"], d["Attachment % (txns)"], marker="o", lw=2.5, color=BR_COL[br], label=BR_FULL[br])
    for x, y in zip(d["year"], d["Attachment % (txns)"]):
        ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 9),
                    ha="center", fontsize=9.5, color=BR_COL[br], fontweight="bold")
ax.set_xticks(list(YR_LBL)); ax.set_xticklabels(YR_LBL.values())
ax.set_ylim(0, 40); ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylabel("Transactions with a Scene+ scan"); ax.set_xlabel("")
ax.legend(frameon=False, loc="upper left")
header(fig, "Scene+ attachment by year", f"Share of transactions with a Scene+ member scanned · {PERIOD}")
fig.tight_layout(rect=[0, 0, 1, 0.88]); plt.show()

# ---------- Q1b: actual Scene+ txns vs 2023-attachment baseline ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, br in zip(axes, ["TRR", "PDM"]):
    d = a[(a["brand"] == br) & (a["year"] >= 2023)].sort_values("year")
    act, flat = d["n_txn_scene"].values, d["Scene+ txns if attachment flat at 2023"].values
    ax.plot(d["year"], act,  marker="o", lw=2.5, color=BR_COL[br], label="Actual Scene+ transactions")
    ax.plot(d["year"], flat, marker="o", lw=2, ls="--", color=GRAY, label="At 2023 attachment rate")
    ax.fill_between(d["year"], flat, act, color=BR_COL[br], alpha=0.12)
    gap = act[-1] - flat[-1]
    ax.annotate(f"+{gap/1e3:,.0f}K from\nhigher attachment", (d["year"].iloc[-1], (act[-1] + flat[-1]) / 2),
                textcoords="offset points", xytext=(-12, 0), ha="right", va="center", fontsize=9.5, color=INK)
    ax.set_xticks(d["year"]); ax.set_xticklabels([YR_LBL[y] for y in d["year"]])
    ax.set_ylim(0, act.max() * 1.15); ax.yaxis.set_major_formatter(kfmt)
    ax.set_title(BR_FULL[br], loc="left", fontweight="bold", color=INK); ax.set_xlabel("")
axes[0].set_ylabel("Scene+ transactions"); axes[0].legend(frameon=False, loc="lower left")
header(fig, "Scene+ transactions: actual vs 2023 attachment rate",
       f"Dashed line = total transactions × 2023 attachment rate · {PERIOD}")
fig.tight_layout(rect=[0, 0, 1, 0.86]); plt.show()

# ---------- Q2a: average visits per member ----------
avg = (tot["visits"] / tot["members"]).rename("avg").reset_index()
fig, ax = plt.subplots(figsize=(10, 5.2))
w = 0.38
for i, br in enumerate(["Rec Room", "Playdium"]):
    d = avg[avg["brand"] == br].sort_values("yr")
    xs = np.arange(len(d)) + (i - 0.5) * w
    bars = ax.bar(xs, d["avg"], width=w, color=BR_COL[br], label=br)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in d["avg"]], padding=3, fontsize=9.5, color=INK)
ax.set_xticks(np.arange(4)); ax.set_xticklabels([YR_LBL[y] for y in [2023, 2024, 2025, 2026]])
ax.set_ylim(0, 2); ax.set_ylabel("Visits per Scene+ member"); ax.set_xlabel("")
ax.legend(frameon=False, loc="upper left", ncol=2)
header(fig, "Scene+ visits per member per year",
       f"Visit = one member, one day, one venue; counted within each brand · {PERIOD}")
fig.tight_layout(rect=[0, 0, 1, 0.88]); plt.show()

# ---------- Q2b: members by visits per year ----------
BINS = ["1", "2-3", "4-6", "7-12", "13+"]
BIN_COL = sns.color_palette("crest", len(BINS))
rows = [(b, y) for b in ["Rec Room", "Playdium"] for y in [2023, 2024, 2025, 2026]]
s = share.reindex(rows)[BINS]
fig, ax = plt.subplots(figsize=(12, 5.6))
ypos = np.arange(len(rows))[::-1] + np.array([0.4 if b == "Rec Room" else 0 for b, _ in rows])
left = np.zeros(len(rows))
for b, c in zip(BINS, BIN_COL):
    vals = s[b].values
    ax.barh(ypos, vals, left=left, color=c, height=0.8, label=f"{b} visit" + ("" if b == "1" else "s"))
    for y, l, v in zip(ypos, left, vals):
        if v >= 4:
            ax.text(l + v / 2, y, f"{v:.1f}%", ha="center", va="center", fontsize=9,
                    color="white" if b == "1" else INK)
    left += vals
ax.set_yticks(ypos); ax.set_yticklabels([f"{b} · {YR_LBL[y]}" for b, y in rows])
ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlabel("Share of Scene+ members")
ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.12))
header(fig, "Scene+ members by number of visits per year",
       f"Labels shown where share ≥ 4% · {PERIOD}")
fig.tight_layout(rect=[0, 0, 1, 0.88]); plt.show()

In [0]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ---------- 1. Pull: Scene+ member visits, max band tier loaded, F&B on that visit ----------
fb_tier = spark.sql("""
WITH id_txn AS (
  SELECT DISTINCT LocationId, txn_date, Transaction_Number, CDE_ID
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v
  WHERE CDE_ID IS NOT NULL AND CDE_ID > 0
    AND YEAR(txn_date) >= 2025 AND DATE_FORMAT(txn_date, 'MM-dd') <= '11-08'
),
txn_agg AS (
  SELECT v.LocationId, v.txn_date, v.Transaction_Number,
         SUM(CASE WHEN v.category = 'F&B' THEN v.Net_Revenue ELSE 0 END) AS fb_rev,
         MAX(CASE WHEN UPPER(i.Item_Description) LIKE '%TAG%'
                   AND UPPER(i.Item_Description) NOT LIKE 'G/E%'
                   AND UPPER(i.Item_Description) NOT LIKE '%FREE%'
                   AND UPPER(i.Item_Description) NOT LIKE '%BOGO%'
                   AND UPPER(i.Item_Description) NOT LIKE '%FOOD%'
                   AND v.Quantity > 0
                THEN TRY_CAST(NULLIF(regexp_extract(UPPER(i.Item_Description), '[$] ?([0-9]+[.]?[0-9]*)', 1), '') AS DOUBLE) END) AS band_usd
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v v
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON v.ItemSkey = i.itemskey
  WHERE YEAR(v.txn_date) >= 2025 AND DATE_FORMAT(v.txn_date, 'MM-dd') <= '11-08'
  GROUP BY 1, 2, 3
),
visit AS (
  SELECT c.LocationId, c.txn_date, c.CDE_ID,
         SUM(a.fb_rev) AS fb_rev, MAX(a.band_usd) AS band_usd
  FROM id_txn c JOIN txn_agg a USING (LocationId, txn_date, Transaction_Number)
  GROUP BY 1, 2, 3
)
SELECT l.LC_Trade_Brand_Description AS brand, YEAR(v.txn_date) AS yr, v.band_usd,
       COUNT(*) AS visits,
       SUM(CASE WHEN v.fb_rev > 0 THEN 1 ELSE 0 END) AS visits_with_fb,
       SUM(v.fb_rev) AS fb_rev
FROM visit v
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON v.LocationId = l.LocationID
GROUP BY 1, 2, 3
""").toPandas()
fb_tier[["visits", "visits_with_fb", "fb_rev"]] = fb_tier[["visits", "visits_with_fb", "fb_rev"]].astype(float)

# ---------- 2. Tier labels (merged bins as in Cell 15) ----------
MERGE = {15: 10, 129: 100}
def tier_label(v):
    if pd.isna(v): return "No band load"
    v = MERGE.get(v, v)
    return {10: "$10–15", 100: "$100–129"}.get(v, f"${v:g}")
fb_tier["tier"] = fb_tier["band_usd"].apply(tier_label)
fb_tier["sort"] = fb_tier["band_usd"].replace(MERGE).fillna(-1)

In [0]:
# ---------- 3. Summary by brand x year x tier ----------
out_y = (fb_tier.groupby(["brand", "yr", "tier", "sort"])[["visits", "visits_with_fb", "fb_rev"]].sum().reset_index()
           .assign(**{"F&B attach %":        lambda x: (100 * x["visits_with_fb"] / x["visits"]).round(1),
                      "F&B $ per visit":     lambda x: (x["fb_rev"] / x["visits"]).round(2),
                      "F&B $ per F&B visit": lambda x: (x["fb_rev"] / x["visits_with_fb"]).round(2)})
           .sort_values(["brand", "yr", "sort"]))

print("V-Q3 visit totals reconcile:", out_y["visits"].sum() == fb_tier["visits"].sum())
print(out_y[["brand", "yr", "tier", "visits", "F&B attach %", "F&B $ per visit", "F&B $ per F&B visit"]].to_string(index=False))

# ---------- 4. Chart: rows = brand, columns = year ----------
BR_COL = {"Rec Room": globals().get("BLUE", "#2a78d6"), "Playdium": globals().get("ORANGE", "#eb6834")}
GREY   = globals().get("GRAY", "#9aa0a6"); INK_ = globals().get("INK", "#1f2933"); MUTED_ = globals().get("MUTED", "#6b7280")
MIN_VISITS = 500
YEARS_Q3 = [2025, 2026]

fig, axes = plt.subplots(2, 2, figsize=(14, 10.5), sharex=True)
xmax = out_y.loc[out_y["visits"] >= MIN_VISITS, "F&B $ per visit"].max() * 1.55
for r, br in enumerate(["Rec Room", "Playdium"]):
    for c, yr in enumerate(YEARS_Q3):
        ax = axes[r, c]
        d = out_y[(out_y["brand"] == br) & (out_y["yr"] == yr) & (out_y["visits"] >= MIN_VISITS)].sort_values("sort")
        y = np.arange(len(d))[::-1]
        ax.barh(y, d["F&B $ per visit"], height=0.7,
                color=[GREY if t == "No band load" else BR_COL[br] for t in d["tier"]])
        for yi, v, att in zip(y, d["F&B $ per visit"], d["F&B attach %"]):
            ax.text(v, yi, f"  ${v:,.2f} · {att:.0f}% buy F&B", va="center", fontsize=9, color=INK_)
        base = d.loc[d["tier"] == "No band load", "F&B $ per visit"]
        if len(base):
            ax.axvline(base.iloc[0], color=GREY, ls="--", lw=1)
        ax.set_yticks(y); ax.set_yticklabels(d["tier"])
        ax.set_xlim(0, xmax); ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
        ax.set_title(f"{br} · {'2026 YTD' if yr == 2026 else yr}", loc="left", fontweight="bold", color=INK_)
        if r == 1: ax.set_xlabel("F&B spend per Scene+ visit")
fig.text(0.01, 0.985, "Scene+ F&B spend per visit, by Game Band tier loaded that visit",
         fontsize=15, fontweight="bold", color=INK_, va="top")
fig.text(0.01, 0.95, f"Grey bar and dashed line = visits with no band load · Tiers with fewer than {MIN_VISITS} visits hidden · "
         "Jan 1 – Nov 8 each year", fontsize=10.5, color=MUTED_, va="top")
fig.tight_layout(rect=[0, 0, 1, 0.92]); plt.show()

In [0]:
nb = spark.sql("""
WITH id_txn AS (
  SELECT DISTINCT LocationId, txn_date, Transaction_Number, CDE_ID
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v
  WHERE CDE_ID IS NOT NULL AND CDE_ID > 0
    AND YEAR(txn_date) >= 2025 AND DATE_FORMAT(txn_date, 'MM-dd') <= '11-08'
),
txn_agg AS (
  SELECT v.LocationId, v.txn_date, v.Transaction_Number,
         MAX(CASE WHEN v.category = 'F&B'   AND v.Net_Revenue > 0 THEN 1 ELSE 0 END) AS has_fb,
         MAX(CASE WHEN v.category = 'Games' AND v.Net_Revenue > 0 THEN 1 ELSE 0 END) AS has_games,
         MAX(CASE WHEN UPPER(i.Item_Description) LIKE '%TAG%'
                   AND UPPER(i.Item_Description) NOT LIKE 'G/E%' AND UPPER(i.Item_Description) NOT LIKE '%FREE%'
                   AND UPPER(i.Item_Description) NOT LIKE '%BOGO%' AND UPPER(i.Item_Description) NOT LIKE '%FOOD%'
                   AND v.Quantity > 0 THEN 1 ELSE 0 END) AS has_band
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v v
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON v.ItemSkey = i.itemskey
  WHERE YEAR(v.txn_date) >= 2025 AND DATE_FORMAT(v.txn_date, 'MM-dd') <= '11-08'
  GROUP BY 1, 2, 3
),
visit AS (
  SELECT c.LocationId, c.txn_date, c.CDE_ID,
         MAX(has_fb) AS has_fb, MAX(has_games) AS has_games, MAX(has_band) AS has_band,
         COUNT(*) AS n_receipts
  FROM id_txn c JOIN txn_agg a USING (LocationId, txn_date, Transaction_Number)
  GROUP BY 1, 2, 3
)
SELECT l.LC_Trade_Brand_Description AS brand,
       CASE WHEN has_band = 1 THEN 'Band load' ELSE 'No band load' END AS grp,
       ROUND(100 * AVG(CASE WHEN has_fb = 1 AND has_games = 0 THEN 1 ELSE 0 END), 1) AS pct_fb_only,
       ROUND(100 * AVG(CASE WHEN has_fb = 0 AND has_games = 1 THEN 1 ELSE 0 END), 1) AS pct_games_only,
       ROUND(100 * AVG(CASE WHEN has_fb = 1 AND has_games = 1 THEN 1 ELSE 0 END), 1) AS pct_both,
       ROUND(100 * AVG(CASE WHEN has_fb = 0 AND has_games = 0 THEN 1 ELSE 0 END), 1) AS pct_neither,
       ROUND(AVG(n_receipts), 2) AS receipts_per_visit
FROM visit v JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON v.LocationId = l.LocationID
GROUP BY 1, 2 ORDER BY 1, 2
""").toPandas()
print(nb.to_string(index=False))

In [0]:
# ============================================================
# Guest behaviour by Game Band tier & location — Scene+ visit base
# Visit = CDE_ID x venue x date, net qty > 0 (same as Visitation Frequency cell)
# Tier  = highest band $ loaded on that visit (same parser as F&B-by-tier cell)
# New   = first-ever visit (lookback to 2022-01-01) falls in the analysis year
# Window: Jan 1 – Nov 8 of 2025 and 2026 (same-period YTD)
# ============================================================
gb_vis = spark.sql("""
WITH id_txn AS (
  SELECT DISTINCT LocationId, txn_date, Transaction_Number, CDE_ID
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v
  WHERE CDE_ID IS NOT NULL AND CDE_ID > 0
),
txn_agg AS (
  SELECT v.LocationId, v.txn_date, v.Transaction_Number,
         SUM(v.Quantity) AS qty,
         MAX(CASE WHEN v.category = 'F&B' AND v.Net_Revenue > 0 THEN 1 ELSE 0 END) AS has_fb,
         MAX(CASE WHEN UPPER(i.Item_Description) LIKE '%TAG%'
                   AND UPPER(i.Item_Description) NOT LIKE 'G/E%' AND UPPER(i.Item_Description) NOT LIKE '%FREE%'
                   AND UPPER(i.Item_Description) NOT LIKE '%BOGO%' AND UPPER(i.Item_Description) NOT LIKE '%FOOD%'
                   AND v.Quantity > 0
                  THEN TRY_CAST(NULLIF(regexp_extract(UPPER(i.Item_Description), '[$] ?([0-9]+[.]?[0-9]*)', 1), '') AS DOUBLE) END) AS band_usd
  FROM data_exploration_hub.marketing_hub.lbe_review_lines_v v
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON v.ItemSkey = i.itemskey
  GROUP BY 1, 2, 3
),
visit AS (
  SELECT c.LocationId, c.txn_date, c.CDE_ID,
         MAX(a.has_fb) AS has_fb, MAX(a.band_usd) AS band_usd
  FROM id_txn c JOIN txn_agg a USING (LocationId, txn_date, Transaction_Number)
  GROUP BY 1, 2, 3
  HAVING SUM(a.qty) > 0
),
v2 AS (
  SELECT v.*, l.LC_Trade_Brand_Description AS brand, l.Location_Name,
         MIN(v.txn_date) OVER (PARTITION BY l.LC_Trade_Brand_Description, v.CDE_ID) AS first_brand_dt,
         MIN(v.txn_date) OVER (PARTITION BY v.LocationId, v.CDE_ID)                 AS first_loc_dt
  FROM visit v JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON v.LocationId = l.LocationID
)
SELECT brand, LocationId, Location_Name, CDE_ID, txn_date, YEAR(txn_date) AS yr, has_fb, band_usd,
       CASE WHEN YEAR(first_brand_dt) = YEAR(txn_date) THEN 1 ELSE 0 END AS new_to_brand,
       CASE WHEN YEAR(first_loc_dt)   = YEAR(txn_date) THEN 1 ELSE 0 END AS new_to_loc
FROM v2
WHERE YEAR(txn_date) IN (2025, 2026) AND DATE_FORMAT(txn_date, 'MM-dd') <= '11-08'
""").toPandas()

# Tier labels (same merges as F&B-by-tier cell)
GB_MERGE = {15: 10, 129: 100}
def gb_tier(v):
    if pd.isna(v): return "No band load"
    v = GB_MERGE.get(v, v)
    return {10: "$10–15", 100: "$100–129"}.get(v, f"${v:g}")
gb_vis["tier_sort"] = gb_vis["band_usd"].replace(GB_MERGE).fillna(-1)
gb_vis["tier"]      = gb_vis["band_usd"].apply(gb_tier)
print(gb_vis.shape)
print(gb_vis.groupby(["brand", "yr"]).agg(visits=("CDE_ID", "size"), members=("CDE_ID", "nunique")))

In [0]:
# ============================================================
# Q1 — By brand x Game Band tier
# Member tier = highest band tier the member loaded at that brand in the year
# (members who never loaded a band = "No band load")
# ============================================================
m = (gb_vis.groupby(["brand", "yr", "CDE_ID"])
          .agg(visits=("txn_date", "size"), fb_visits=("has_fb", "sum"),
               tier_sort=("tier_sort", "max"), new=("new_to_brand", "max"))
          .reset_index())
m["tier"] = m["tier_sort"].map(lambda s: gb_tier(np.nan if s < 0 else s))

q1 = (m.groupby(["brand", "yr", "tier_sort", "tier"])
       .agg(members=("CDE_ID", "size"), visits=("visits", "sum"), fb_visits=("fb_visits", "sum"),
            fb_members=("fb_visits", lambda s: (s > 0).sum()), new_members=("new", "sum"),
            repeat_members=("visits", lambda s: (s > 1).sum()))
       .reset_index())
q1 = q1.assign(**{
    "Member share %":       lambda x: (100 * x["members"] / x.groupby(["brand", "yr"])["members"].transform("sum")).round(1),
    "Visits / member":      lambda x: (x["visits"] / x["members"]).round(2),
    "% repeat (2+ visits)": lambda x: (100 * x["repeat_members"] / x["members"]).round(1),
    "% members buying F&B": lambda x: (100 * x["fb_members"] / x["members"]).round(1),
    "F&B attach % (visits)": lambda x: (100 * x["fb_visits"] / x["visits"]).round(1),
    "% new":                lambda x: (100 * x["new_members"] / x["members"]).round(1),
    "% returning":          lambda x: (100 - 100 * x["new_members"] / x["members"]).round(1),
}).sort_values(["brand", "yr", "tier_sort"])

# Validation: members reconcile to distinct members per brand-year
chk = gb_vis.groupby(["brand", "yr"])["CDE_ID"].nunique()
print("Members reconcile:", (q1.groupby(["brand", "yr"])["members"].sum() == chk).all())

Q1_COLS = ["brand", "yr", "tier", "members", "Member share %", "Visits / member", "% repeat (2+ visits)",
           "% members buying F&B", "F&B attach % (visits)", "% new", "% returning"]
display(q1[Q1_COLS])

In [0]:
# ============================================================
# Q2 — By location (visits, band tier mix, F&B attach, new vs return)
# Frequency = visits per member AT that location
# Tier mix  = share of band-load visits by tier (+ % of visits with any band load)
# New       = first-ever visit to that location (and to the brand) in the year
# ============================================================
lm = (gb_vis.groupby(["brand", "Location_Name", "yr", "CDE_ID"])
           .agg(visits=("txn_date", "size"), new_loc=("new_to_loc", "max"), new_brand=("new_to_brand", "max"))
           .reset_index())
q2 = (lm.groupby(["brand", "Location_Name", "yr"])
        .agg(members=("CDE_ID", "size"), visits=("visits", "sum"),
             new_loc=("new_loc", "sum"), new_brand=("new_brand", "sum"))
        .reset_index())
vv = (gb_vis.assign(band=gb_vis["tier_sort"] >= 0)
            .groupby(["brand", "Location_Name", "yr"])
            .agg(fb_visits=("has_fb", "sum"), band_visits=("band", "sum")).reset_index())
q2 = q2.merge(vv, on=["brand", "Location_Name", "yr"])
q2 = q2.assign(**{
    "Visits / member":       (q2["visits"] / q2["members"]).round(2),
    "F&B attach % (visits)": (100 * q2["fb_visits"] / q2["visits"]).round(1),
    "% visits w/ band":      (100 * q2["band_visits"] / q2["visits"]).round(1),
    "% new to location":     (100 * q2["new_loc"] / q2["members"]).round(1),
    "% new to brand":        (100 * q2["new_brand"] / q2["members"]).round(1),
    "% returning":           (100 - 100 * q2["new_loc"] / q2["members"]).round(1),
})

# Tier mix among band-load visits (main tiers; $55 and odd values -> Other)
MAIN_TIERS = ["$10–15", "$25", "$35", "$50", "$75", "$100–129"]
bv = gb_vis[gb_vis["tier_sort"] >= 0].copy()
bv["tier_g"] = np.where(bv["tier"].isin(MAIN_TIERS), bv["tier"], "Other")
mix = (pd.crosstab([bv["brand"], bv["Location_Name"], bv["yr"]], bv["tier_g"], normalize="index") * 100).round(1)
mix = mix.reindex(columns=[c for c in MAIN_TIERS + ["Other"] if c in mix.columns]).add_prefix("mix ").reset_index()
q2 = q2.merge(mix, on=["brand", "Location_Name", "yr"], how="left")

# Validation: visits reconcile to visit base
print("Visits reconcile:", q2["visits"].sum() == len(gb_vis))

Q2_COLS = (["brand", "Location_Name", "yr", "members", "Visits / member", "F&B attach % (visits)", "% visits w/ band"]
           + [c for c in q2.columns if c.startswith("mix ")] + ["% new to location", "% new to brand", "% returning"])
display(q2.sort_values(["brand", "yr", "members"], ascending=[True, True, False])[Q2_COLS])

## Guest Behaviour by Game Band Tier & Location — Summary
**Scope:** Scene+ identified guests only, Jan 1 – Nov 8 of 2025 and 2026 YTD. A visit is one guest at one venue on one day. A guest's tier is the highest band they loaded at that brand in the year. A guest is **new** if their first visit since Jan 2022 falls in that year.

### 1. By Game Band tier — a $100 guest does not behave like a $50 guest
| Rec Room 2025 | No band | \$10–15 | \$25 | \$35 | \$50 | \$75 | \$100–129 |
|---|---|---|---|---|---|---|---|
| Share of members | 17.6% | 3.0% | 17.3% | 17.4% | 24.1% | 8.2% | 12.4% |
| Visits / member | 1.16 | 1.12 | 1.19 | 1.30 | 1.35 | 1.42 | **1.70** |
| % with 2+ visits | 8.3% | 7.6% | 13.0% | 18.7% | 21.3% | 24.1% | **30.9%** |
| % members buying F&B | 89.0% | 15.2% | 13.0% | 15.9% | 17.5% | 19.6% | 23.2% |
| % new / returning | 72 / 28 | 77 / 23 | 71 / 29 | 67 / 33 | 67 / 33 | 68 / 32 | **62 / 38** |

| Playdium 2025 | No band | \$10–15 | \$25 | \$35 | \$50 | \$75 | \$100–129 |
|---|---|---|---|---|---|---|---|
| Share of members | 24.3% | 6.6% | 13.5% | 14.9% | 24.7% | 6.4% | 9.6% |
| Visits / member | 1.15 | 1.31 | 1.19 | 1.27 | 1.35 | 1.39 | **1.68** |
| % with 2+ visits | 8.4% | 18.1% | 12.3% | 17.0% | 21.5% | 23.8% | **30.8%** |
| % members buying F&B | 47.2% | 67.9% | 14.5% | 18.2% | 21.0% | 24.4% | 26.4% |
| % new / returning | 72 / 28 | 81 / 19 | 75 / 25 | 70 / 30 | 68 / 32 | 68 / 32 | 68 / 32 |

* **Visit frequency rises steadily with tier.** The overall average of about 1.3 hides a range from 1.1–1.2 visits (no band to \$25) up to **1.7 at \$100–129**. A \$100 guest visits about 25% more often than a \$50 guest (1.70 vs 1.35) and is about 1.45x as likely to come back within the year (31% vs 21%). The pattern is almost the same for both brands and repeats in 2026 YTD (TRR 1.15 → 1.64, PDM 1.13 → 1.58).
* **F&B attachment is low for band loaders and rises with tier.** Among band loaders, 13–26% buy F&B; the higher the tier, the more likely. F&B buyers are mostly guests who **don't load a band** (TRR 89%, i.e. diners). Playdium's \$10–15 tier is the exception: 68% buy F&B, which suggests a small band added to a meal or party.
* **Higher tiers have more returning guests.** About two-thirds of guests in every tier are new. Returning share goes from 23–29% (\$10–\$25) to 38% at TRR \$100–129, and the gap gets wider in 2026 (TRR \$100–129: 45% returning). Low tiers work as the way in for first-time guests. The \$100+ tier is the repeat-player core.

### 2. By location — venues fall into distinct behaviour types
* **Local "eat + play" repeat venues:** Masonville (1.41 visits/member, **42% F&B attach**, 64% band loaders, 32% returning), Park Place Barrie (1.30, 39%), Avalon Mall (**1.62**, 35%, **56% returning**, mostly \$25–35 bands with only 6% at \$100+), Seasons of Tuxedo (1.32, 33%, 39% returning).
* **Destination, games-first, first-time-heavy venues:** Square One (1.19, **18% F&B**, 88% band loaders, only **17% returning**), Roundhouse (1.12, 85% new), Granville (12.5% F&B, 93% band loaders), Royal Mount (10% F&B, 95% new to the venue).
* **High-tier skew:** Deerfoot (\$100–129 = 22% of band visits, \$75 = 12%) and Brentwood (19%) lean premium. Avalon Mall and Royal Mount lean to entry tiers (\$25 = 37% and 29%).
* **Square One vs Masonville (2025):** Masonville guests visit more often (1.41 vs 1.19) and are more than twice as likely to buy F&B (42% vs 18%). They load bands less (64% vs 88%) and are almost twice as likely to be returning (32% vs 17%). Square One is a games-first venue where most guests visit once. Masonville is a local venue where guests eat, play and come back. Tier mix is similar at both (\$100–129 = 16% vs 14%).
* **Playdium:** Dartmouth has the most repeat guests (1.30 visits, 39% returning, \$50 = 36% of band visits). Whitby has the highest F&B attach (35%). Orion Gate has the lowest frequency (1.17). Fairview (opened Dec 2024) is 97% new in 2025 and skews to \$10–15 bands (18%).

### Caveats
* Results cover Scene+ guests only (about 34% of PDM and fewer TRR transactions). Non-members may behave differently.
* "New" uses a lookback to Jan 2022. Venues that opened or reopened recently (Fairview, Vaughan 2026, and likely Granville and Royal Mount, where 95%+ of guests are new to the venue) show inflated new-guest shares and should not be compared on new vs returning.
* F&B attach counts any F&B purchase by the guest that day, across all their receipts. The \$55 tier only appears in 2026 and is grouped under "Other" in the location mix.